# Figure 2 — Distinct perturbation programs converge on failed endothelial differentiation

This notebook reproduces panels **a, b, d, e and f**. Panel c is intentionally omitted because it is a schematic.

- **Panels a–b:** use the attached cNMF workbook copied into the repository at `data/figure_02/cNMF_50_0_2_032526_6co.xlsx`. The notebook reads the `Perturbation Association 1` sheet by a relative repository path.
- **Panel d:** recalculates one D3 EC percentage per guide from the integrated AnnData guide assignments. For each non-targeting guide, cells carrying a guide targeting **KDR, PLCG1, PRKCD or ETV2** are excluded. Each target is compared with the resulting non-targeting-guide distribution using a two-sided Welch t-test; P values are BH-adjusted across the four displayed targets.
- **Panel e:** follows the prior UMAP style, including the `YlGnBu` color map, zero-valued gray background and feature-specific 99th-percentile maxima.
- **Panel f:** retains the carrier-seeded neighborhood implementation.

For panel b's rightmost EC-output column, the same guide-level recalculation is generalized to the 14 displayed genes; non-targeting-guide cells carrying any of those 14 targets are excluded, and Welch P values are BH-adjusted across the 14 displayed genes.


In [ ]:
from __future__ import annotations

from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from src.cc_perturbseq.figure_notebook import (
    get_path, initialize_figure, load_figure_manifest, panel_directory,
    sha256_file, write_receipt,
)
from src.cc_perturbseq.figure2_panels import (
    FIGURE2D_TARGETS, FIGURE2_PROGRAMS, FIGURE2_TARGET_GROUPS,
    compute_d3_guide_ec_statistics, load_figure2_cnmf_workbook,
    plot_figure2a, plot_figure2b, plot_figure2d, plot_figure2e_umaps,
)
from src.cc_perturbseq.program_panels import (
    carrier_seeded_neighborhood, load_guide_target_map,
    load_integrated_anndata, plot_carrier_neighborhood_grid,
)

EXPECTED_SOURCE_WORKBOOK_SHA256 = "d2fe4834b1efd3c00e01062b56b907a6178bfbc48a40beb5ea7b347538bfc824"


def export_figure(fig, panel: str, stem: str, *, png_dpi: int = 300):
    outdir = panel_directory(ctx, panel)
    outputs = {}
    for suffix in ("pdf", "svg", "png"):
        path = outdir / f"{stem}.{suffix}"
        fig.savefig(
            path,
            dpi=png_dpi if suffix == "png" else None,
            bbox_inches="tight",
            pad_inches=0.03,
            facecolor="white",
        )
        outputs[suffix] = str(path)
    return outputs


FIGURE = load_figure_manifest("Figure_02")
FIGURE["panels"] = [
    panel for panel in FIGURE["panels"]
    if panel["panel"] in {"a", "b", "d", "e", "f"}
]
for panel in FIGURE["panels"]:
    if panel["panel"] == "a":
        panel["required_inputs"] = []
    elif panel["panel"] in {"b", "d", "e", "f"}:
        panel["required_inputs"] = [
            "ccperturbseq.integrated_anndata",
            "ccperturbseq.cnmf_usage",
            "ccperturbseq.target_metadata",
        ]
FIGURE["scope"] = "Panels a–b and d–f"
ctx = initialize_figure(FIGURE)
ctx["input_audit"]


## Shared inputs

Panels a and b read the original supplied cNMF workbook from a relative path under `data/figure_02/`. Single-cell panels use the final integrated AnnData, K = 50 cNMF usage table and guide metadata configured for Sherlock.


In [ ]:
cnmf_workbook = ctx["root"] / "data" / "figure_02" / "cNMF_50_0_2_032526_6co.xlsx"
if sha256_file(cnmf_workbook) != EXPECTED_SOURCE_WORKBOOK_SHA256:
    raise ValueError("Repository cNMF workbook does not match the supplied source workbook SHA-256.")
effect_matrix, selected_associations, workbook_provenance = load_figure2_cnmf_workbook(cnmf_workbook)
provenance_map = dict(zip(workbook_provenance["field"].astype(str), workbook_provenance["value"].astype(str)))

adata_path = get_path(ctx["paths"], "ccperturbseq.integrated_anndata")
usage_path = get_path(ctx["paths"], "ccperturbseq.cnmf_usage")
guide_metadata_path = get_path(ctx["paths"], "ccperturbseq.target_metadata")
adata, usage = load_integrated_anndata(adata_path, usage_path)

{
    "cNMF_workbook": str(cnmf_workbook),
    "cNMF_workbook_sha256": sha256_file(cnmf_workbook),
    "effect_matrix_shape": effect_matrix.shape,
    "selected_association_rows": len(selected_associations),
    "single_cells": adata.n_obs,
    "programs": usage.shape[1],
}


## Panel a — pairwise correlation of perturbation effects

The matrix contains 200 features per profile: 50 cNMF programs at each of D0, D1, D2 and D3. The duplicate alias `NEDD8-MDP1` is removed and the pooled `non-targeting` profile is retained, giving 297 canonical target genes plus one pooled non-targeting control. Pairwise Pearson correlations are ordered by average-linkage clustering of distance \(1-r\) with optimal leaf ordering.


In [ ]:
fig_a, correlation_a, order_a, inset_a, boxes_a = plot_figure2a(effect_matrix)
out_a = panel_directory(ctx, "a")
outputs_a = export_figure(fig_a, "a", "Figure_02a_perturbation_effect_correlation")
correlation_a.to_csv(out_a / "Figure_02a_target_correlation.csv")
order_a.to_csv(out_a / "Figure_02a_clustered_target_order.csv", index=False)
inset_a.to_csv(out_a / "Figure_02a_selected_group_correlations.csv", index=False)
boxes_a.to_csv(out_a / "Figure_02a_cluster_boxes.csv", index=False)
write_receipt(
    ctx, "a", status="SOURCE_VERIFIED_IMPLEMENTATION",
    inputs={
        "repository_workbook": str(cnmf_workbook),
        "repository_workbook_sha256": sha256_file(cnmf_workbook),
        "supplied_source_workbook_sha256": EXPECTED_SOURCE_WORKBOOK_SHA256,
    },
    outputs={
        **outputs_a,
        "correlation": str(out_a / "Figure_02a_target_correlation.csv"),
        "target_order": str(out_a / "Figure_02a_clustered_target_order.csv"),
    },
    parameters={
        "profiles": "297 canonical target genes plus pooled non-targeting control",
        "features": "D0–D3 × P0–P49 (200)",
        "correlation": "Pearson",
        "distance": "1 - Pearson r",
        "linkage": "average with optimal ordering",
        "duplicate_alias_removed": "NEDD8-MDP1",
    },
)
plt.show()


## Panel b — selected regulator effects on programs and EC output

Program effects and adjusted P values are read from the repository-local cNMF workbook. The rightmost EC-output column is recalculated at the guide level from D3 cells for the 14 displayed genes. For each non-targeting guide, cells carrying any guide targeting one of those 14 genes are excluded. EC-output significance uses two-sided Welch t-tests and BH correction across the 14 displayed genes.


In [ ]:
panel_b_targets = [gene for genes in FIGURE2_TARGET_GROUPS.values() for gene in genes]
guide_b, stats_b, metadata_b = compute_d3_guide_ec_statistics(
    adata,
    guide_metadata_path,
    panel_b_targets,
    nt_exclusion_targets=panel_b_targets,
)
fig_b, effects_b, qvalues_b, program_source_b, ec_source_b = plot_figure2b(
    selected_associations,
    stats_b,
)
out_b = panel_directory(ctx, "b")
outputs_b = export_figure(fig_b, "b", "Figure_02b_selected_program_and_EC_effects")
effects_b.to_csv(out_b / "Figure_02b_effect_matrix.csv")
qvalues_b.to_csv(out_b / "Figure_02b_FDR_matrix.csv")
program_source_b.to_csv(out_b / "Figure_02b_program_effect_source.csv", index=False)
guide_b.to_csv(out_b / "Figure_02b_guide_level_D3_EC_percentages.csv", index=False)
ec_source_b.to_csv(out_b / "Figure_02b_Welch_EC_statistics.csv", index=False)
write_receipt(
    ctx, "b", status="IMPLEMENTED_WITH_EXPLICIT_EC_ASSUMPTION",
    inputs={
        "repository_workbook": str(cnmf_workbook),
        "integrated_anndata": str(adata_path),
        "cnmf_usage": str(usage_path),
        "guide_metadata": str(guide_metadata_path),
    },
    outputs={
        **outputs_b,
        "effect_matrix": str(out_b / "Figure_02b_effect_matrix.csv"),
        "program_effect_source": str(out_b / "Figure_02b_program_effect_source.csv"),
        "guide_level_EC": str(out_b / "Figure_02b_guide_level_D3_EC_percentages.csv"),
        "EC_statistics": str(out_b / "Figure_02b_Welch_EC_statistics.csv"),
    },
    parameters={
        "program_effect": "approx_log2FC",
        "program_significance": "adj_pval from Perturbation Association 1",
        "EC_effect": "log2(mean target-guide EC% / mean non-targeting-guide EC%)",
        "EC_test": "two-sided Welch t-test",
        "EC_BH_tests": len(panel_b_targets),
        "NT_exclusion_targets": panel_b_targets,
        **metadata_b,
    },
)
plt.show()
stats_b[["gene", "mean_target_EC_pct", "mean_non_targeting_EC_pct", "welch_pvalue_two_sided", "FDR_BH", "significance"]]


## Panel d — guide-level Day 3 EC differentiation efficiency

For each of KDR, PLCG1, PRKCD and ETV2, each point is the D3 EC percentage among carriers of one targeting guide. For every individual non-targeting guide, cells carrying any guide targeting one of these four genes are excluded. Targeting-guide percentages are otherwise calculated from all D3 carriers of that guide. Welch P values are BH-adjusted across the four displayed target genes.


In [ ]:
guide_d, stats_d, metadata_d = compute_d3_guide_ec_statistics(
    adata,
    guide_metadata_path,
    FIGURE2D_TARGETS,
    nt_exclusion_targets=FIGURE2D_TARGETS,
)
fig_d, jitter_d = plot_figure2d(guide_d, stats_d)
out_d = panel_directory(ctx, "d")
outputs_d = export_figure(fig_d, "d", "Figure_02d_guide_level_D3_EC_fraction")
guide_d.to_csv(out_d / "Figure_02d_guide_level_D3_EC_percentages.csv", index=False)
stats_d.to_csv(out_d / "Figure_02d_Welch_statistics.csv", index=False)
jitter_d.to_csv(out_d / "Figure_02d_plot_jitter_coordinates.csv", index=False)
write_receipt(
    ctx, "d", status="SOURCE_VERIFIED_METHOD_IMPLEMENTATION",
    inputs={
        "integrated_anndata": str(adata_path),
        "guide_metadata": str(guide_metadata_path),
    },
    outputs={
        **outputs_d,
        "guide_level_EC": str(out_d / "Figure_02d_guide_level_D3_EC_percentages.csv"),
        "Welch_statistics": str(out_d / "Figure_02d_Welch_statistics.csv"),
    },
    parameters={
        "targets": FIGURE2D_TARGETS,
        "NT_exclusion_targets": FIGURE2D_TARGETS,
        "test": "two-sided Welch t-test",
        "BH_tests": len(FIGURE2D_TARGETS),
        "unit": "guide-level D3 EC percentage",
        **metadata_d,
    },
)
plt.show()
stats_d[["gene", "n_targeting_guides", "n_non_targeting_guides", "mean_target_EC_pct", "FDR_BH", "significance"]]


## Panel e — P17/P27 and marker UMAPs

This restores the prior continuous-UMAP style: gray background cells, positive-valued cells layered in increasing value order, `YlGnBu`, `vmin = 0`, and a feature-specific 99th-percentile maximum.


In [ ]:
fig_e, source_e = plot_figure2e_umaps(adata, usage)
out_e = panel_directory(ctx, "e")
outputs_e = export_figure(fig_e, "e", "Figure_02e_P17_P27_SLC8A1_GATA4_umaps")
source_e_paths = {}
for feature_name, table in source_e.items():
    path = out_e / f"Figure_02e_{feature_name}_source.csv.gz"
    table.to_csv(path, compression="gzip")
    source_e_paths[feature_name] = str(path)
write_receipt(
    ctx, "e", status="SOURCE_STYLE_RECOVERED",
    inputs={"integrated_anndata": str(adata_path), "cnmf_usage": str(usage_path)},
    outputs={**outputs_e, **source_e_paths},
    parameters={
        "features": ["P17", "SLC8A1", "P27", "GATA4"],
        "cmap": "YlGnBu",
        "vmin": 0,
        "vmax": "feature-specific p99",
        "background": "gray cells at value 0",
    },
)
plt.show()


## Panel f — carrier-seeded neighborhood enrichment

The existing same-day, 300-neighbor program-space implementation is retained.


In [ ]:
guide_map = load_guide_target_map(adata, guide_metadata_path)
targets_f = ["ETV2", "KDR", "PRKCD", "PLCG1"]
neighborhoods_f = carrier_seeded_neighborhood(
    adata, usage, guide_map, targets_f, k=300, pseudocount=0.5,
)
fig_f = plot_carrier_neighborhood_grid(adata, neighborhoods_f, targets_f)
out_f = panel_directory(ctx, "f")
outputs_f = export_figure(fig_f, "f", "Figure_02f_carrier_seeded_neighborhoods")
neighborhoods_f.to_csv(out_f / "Figure_02f_neighborhood_statistics.csv", index=False)
write_receipt(
    ctx, "f", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED",
    inputs={
        "integrated_anndata": str(adata_path),
        "cnmf_usage": str(usage_path),
        "guide_metadata": str(guide_metadata_path),
    },
    outputs={
        **outputs_f,
        "neighborhood_statistics": str(out_f / "Figure_02f_neighborhood_statistics.csv"),
    },
    parameters={
        "targets": targets_f,
        "k": 300,
        "distance": "Euclidean on z-scored 50-program usage",
        "same_day": True,
        "pseudocount": 0.5,
    },
)
plt.show()
